# GVH Diagonal Cubic 0.3.2.7.3.7.3 — Full Hypersurface Constraint Algebra Closure

**Auteur :** Charlemagne O Laurince

## Mission

Tester directement le verrou suivant :
\[
\boxed{R_{DD3}=0\ ?}
\]

à partir des densités canoniques obtenues dans la chaîne `7.7.2`.

Les crochets cibles sont :
\[
\{\mathcal C_i,\mathcal C_j\},\qquad
\{\mathcal C_\perp,\mathcal C_i\},\qquad
\{\mathcal C_\perp,\mathcal C_\perp\}.
\]

### Règle d'audit

Avant de calculer l'algèbre de Dirac, il faut vérifier que
\[
\mathcal C_\perp
\]
est réellement une **densité secondaire indépendante du multiplicateur lapse \(N\)**.

Si la forme obtenue contient encore
\[
a_i^{(n)}=D_i\ln N,
\]
alors la variation correcte par rapport à \(N\) est une variation fonctionnelle d'Euler–Lagrange :
\[
\frac{\delta H}{\delta N}
=
\frac{\partial H}{\partial N}
-
D_i\!\left(
\frac{\partial H}{\partial(D_iN)}
\right),
\]
et non une simple dérivée algébrique.

Le notebook ne déclarera donc pas `hypersurface_algebra_closed=True` tant que ce prérequis n'est pas satisfait.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [1]:
from __future__ import annotations
import sympy as sp, json, sys
from pathlib import Path

print("GVH 0.3.2.7.3.7.3")
print("Python:",sys.version.split()[0])
print("SymPy:",sp.__version__)


GVH 0.3.2.7.3.7.3
Python: 3.12.13
SymPy: 1.14.0


## 1. Reconstruction du secteur spatial de 7.7.2.4

On reconstruit exactement
\[
\mathcal L_u
=
\frac12V^TQ_uV+J_u^TV+U_u
\]
et on teste explicitement si \(J_u\) ou \(U_u\) dépendent de
\[
a_i^{(n)}.
\]


In [2]:
c1,c2,c3,c4 = sp.symbols("c1 c2 c3 c4", real=True)
s = sp.symbols("s", real=True)
v = sp.Matrix(sp.symbols("v1:4", real=True))

K11,K22,K33,K12,K13,K23 = sp.symbols("K11 K22 K33 K12 K13 K23", real=True)
K = sp.Matrix([[K11,K12,K13],[K12,K22,K23],[K13,K23,K33]])
Sdot = sp.symbols("Sdot", real=True)
Vdot = sp.Matrix(sp.symbols("Vdot1:4", real=True))
aN = sp.Matrix(sp.symbols("aN1:4", real=True))     # a_i^(n)=D_i ln N
Gs = sp.Matrix(sp.symbols("Gs1:4", real=True))     # D_i s
Qv = sp.Matrix(3,3,sp.symbols("Q11 Q12 Q13 Q21 Q22 Q23 Q31 Q32 Q33", real=True))

A = sp.expand(-Sdot-v.dot(aN))
B = sp.expand(s*aN+Vdot-K*v)
C = sp.expand(-Gs-K*v)
D = sp.expand(Qv+s*K)

I1 = sp.expand(A**2-B.dot(B)-C.dot(C)+sum(D[i,j]**2 for i in range(3) for j in range(3)))
theta = sp.expand(-A+sp.trace(D))
I3 = sp.expand(A**2-2*B.dot(C)+sum(D[i,j]*D[j,i] for i in range(3) for j in range(3)))
alpha = sp.expand(s*A+v.dot(C))
beta = sp.expand(s*B+D.T*v)
a2 = sp.expand(-alpha**2+beta.dot(beta))
Lu = sp.expand(-c1*I1-c2*theta**2-c3*I3+c4*a2)

vel = sp.Matrix([K11,K22,K33,K12,K13,K23,Sdot,Vdot[0],Vdot[1],Vdot[2]])
zero_vel={x:0 for x in vel}

Qu=sp.hessian(Lu,list(vel))
Ju=sp.Matrix([sp.simplify(sp.diff(Lu,x).subs(zero_vel)) for x in vel])
Uu=sp.simplify(Lu.subs(zero_vel))

assert sp.expand(Lu-(sp.Rational(1,2)*(vel.T*Qu*vel)[0]+(Ju.T*vel)[0]+Uu))==0
print("7.7.2.4 velocity/space split reconstructed: PASS")


7.7.2.4 velocity/space split reconstructed: PASS


In [3]:
U_dep = [bool(Uu.has(x)) for x in aN]
J_dep = [i for i,e in enumerate(Ju) if any(e.has(x) for x in aN)]

print("U_u depends on a_i^(n):",U_dep)
print("J_u components depending on a_i^(n):",J_dep)

assert any(U_dep)
assert len(J_dep)>0
print("LAPSE-GRADIENT DEPENDENCE DETECTED: PASS")


U_u depends on a_i^(n): [True, True, True]
J_u components depending on a_i^(n): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
LAPSE-GRADIENT DEPENDENCE DETECTED: PASS


## 2. Conséquence canonique

Comme
\[
a_i^{(n)}=D_i\ln N=\frac{D_iN}{N},
\]
le Hamiltonien reconstruit à partir de
\[
Q,\ J,\ U
\]
contient encore \(D_iN\).

Ainsi, l'expression
\[
\frac12(P-J)^TQ^{-1}(P-J)-U
\]
est bien la transformée de Legendre du secteur dynamique, mais **elle ne suffit pas encore à identifier la contrainte secondaire normale** par une simple dérivée en \(N\).

Il faut d'abord calculer :
\[
\boxed{
\mathscr C_N
\equiv
-\frac{\delta H_C}{\delta N}
=
-\frac{\partial H_C}{\partial N}
+
D_i\!\left[
\frac{\partial H_C}{\partial(D_iN)}
\right].
}
\]

C'est cette densité \(\mathscr C_N\), après élimination des dépendances de multiplicateur et réduction éventuelle par les contraintes auxiliaires, qui doit entrer dans l'algèbre hypersurface.


## 3. Secteur momentum–momentum

La densité de shift issue de 7.7.2.4 est
\[
\mathcal C_i
=
-2h_{ij}D_k\pi^{kj}
+p_sD_is
+p_v^{\,j}D_iv_j
-D_j(p_v^{\,j}v_i).
\]

Sous forme smeared,
\[
D[\xi]=\int d^3x\,\xi^i\mathcal C_i,
\]
elle est équivalente, modulo terme de bord, au générateur canonique de difféomorphismes spatiaux :
\[
D[\xi]
=
\int d^3x
\left(
\pi^{ij}\mathcal L_\xi h_{ij}
+p_s\mathcal L_\xi s
+p_v^{\,i}\mathcal L_\xi v_i
\right).
\]

Il en résulte structurellement :
\[
\boxed{
\{D[\xi],D[\eta]\}
=
D[[\xi,\eta]].
}
\]


In [4]:
# Algebraic Lie-bracket consistency check on polynomial vector fields in R^3.
x,y,z = sp.symbols("x y z", real=True)
coords=(x,y,z)

xi=sp.Matrix([x*y, y+z, z*x])
eta=sp.Matrix([x+z, x*y, y*z])
zet=sp.Matrix([y*z, x-z, x*y+z])

def lie(X,Y):
    return sp.Matrix([
        sp.expand(sum(X[j]*sp.diff(Y[i],coords[j])-Y[j]*sp.diff(X[i],coords[j]) for j in range(3)))
        for i in range(3)
    ])

jac=sp.simplify(lie(xi,lie(eta,zet))+lie(eta,lie(zet,xi))+lie(zet,lie(xi,eta)))
assert jac==sp.zeros(3,1)

print("Spatial Lie-bracket Jacobi identity: PASS")
print("{D[xi],D[eta]} = D[[xi,eta]] structurally registered")


Spatial Lie-bracket Jacobi identity: PASS
{D[xi],D[eta]} = D[[xi,eta]] structurally registered


## 4. Secteur normal–momentum

Si la contrainte normale finale \(\mathscr C_N\) est une densité scalaire spatiale de poids 1, alors
\[
\boxed{
\{H[N],D[\xi]\}
=
-H[\mathcal L_\xi N].
}
\]

La covariance spatiale de l'action candidate rend cette structure attendue.

Mais le test ne peut être marqué **full explicit** tant que \(\mathscr C_N\) n'est pas extraite de la variation fonctionnelle complète en \(N\).


## 5. Secteur normal–normal

L'identité ADM standard attendue, éventuellement modulo contraintes supplémentaires du secteur directionnel, est
\[
\{H[N],H[M]\}
=
D\!\left[h^{ij}(N\partial_jM-M\partial_jN)\right]
+\text{termes proportionnels à contraintes auxiliaires}.
\]

Pour vérifier cette égalité, il faut disposer de la vraie densité
\[
\mathscr C_N=-\delta H_C/\delta N
\]
et calculer ses dérivées fonctionnelles par rapport aux paires canoniques.

Comme la forme actuelle contient encore \(D_iN\), effectuer maintenant
\[
\{\mathcal C_\perp,\mathcal C_\perp\}
\]
avec la forme de 7.7.2.6 confondrait une transformée de Legendre avec la contrainte secondaire lapse finale.

Le calcul est donc **bloqué méthodologiquement**, pas déclaré faux.


## 6. Révision causale de la chaîne 7.7.2

Le résultat `RDD2_computed=True` de 7.7.2.6 doit être interprété plus précisément :

\[
\boxed{
R_{DD2}^{\rm Legendre}=0
}
\]
sur la branche générique non dégénérée : l'inverse et la transformée de Legendre sont fermées.

Mais la composante
\[
R_{DD2}^{\rm lapse\;functional}
\]
n'était pas encore séparée explicitement.

Le présent audit découvre donc un **nouveau verrou de canonicalisation du lapse**, nécessaire avant la fermeture de \(R_{DD3}\).

Cela ne réfute pas les inversions de 7.7.2.6 ; cela affine la frontière entre :
\[
\text{Hamiltonien canonique}
\quad\text{et}\quad
\text{contrainte secondaire obtenue par }\delta/\delta N.
\]


In [5]:
GATES={
    "Ci_generator_canonical_form_available":True,
    "Ci_Cj_spatial_diffeomorphism_closure_structural":True,
    "spatial_Lie_Jacobi_verified":True,

    "lapse_gradient_dependence_in_U_detected":True,
    "lapse_gradient_dependence_in_J_detected":True,
    "true_secondary_normal_constraint_extracted":False,
    "Cperp_Ci_explicit_functional_bracket":False,
    "Cperp_Cperp_explicit_functional_bracket":False,
    "auxiliary_constraint_terms_classified":False,
    "RDD3_computed":False,
    "RDD3_closed":False,
    "hypersurface_algebra_closed":False,
}

for k,vv in GATES.items():
    print(k,":",vv)

FINAL_STATUS=(
    "PARTIAL-PASS-MOMENTUM-SECTOR-DIFFEOMORPHISM-CLOSURE_"
    "NEW-LAPSE-GRADIENT-FUNCTIONAL-VARIATION-OBSTRUCTION-DETECTED_"
    "BLOCKED-TRUE-NORMAL-SECONDARY-CONSTRAINT-AND-HH-BRACKET"
)
DISPERSION_READY=False

assert GATES["lapse_gradient_dependence_in_U_detected"]
assert not GATES["RDD3_computed"]
assert not GATES["hypersurface_algebra_closed"]
assert DISPERSION_READY is False

print("\nFINAL STATUS:",FINAL_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


Ci_generator_canonical_form_available : True
Ci_Cj_spatial_diffeomorphism_closure_structural : True
spatial_Lie_Jacobi_verified : True
lapse_gradient_dependence_in_U_detected : True
lapse_gradient_dependence_in_J_detected : True
true_secondary_normal_constraint_extracted : False
Cperp_Ci_explicit_functional_bracket : False
Cperp_Cperp_explicit_functional_bracket : False
auxiliary_constraint_terms_classified : False
RDD3_computed : False
RDD3_closed : False
hypersurface_algebra_closed : False

FINAL STATUS: PARTIAL-PASS-MOMENTUM-SECTOR-DIFFEOMORPHISM-CLOSURE_NEW-LAPSE-GRADIENT-FUNCTIONAL-VARIATION-OBSTRUCTION-DETECTED_BLOCKED-TRUE-NORMAL-SECONDARY-CONSTRAINT-AND-HH-BRACKET
DISPERSION_READY = False


## 7. Prochaine étape requise avant de reprendre 7.7.3

### `0.3.2.7.3.7.3.1 — Lapse-Gradient Canonicalization and True Secondary Normal Constraint Extraction`

Cette étape devra :

1. réintroduire explicitement
   \[
   a_i^{(n)}=\frac{D_iN}{N};
   \]

2. écrire \(H_C[N]\) avec toute sa dépendance en
   \[
   N,\quad D_iN;
   \]

3. calculer
   \[
   \boxed{
   \mathscr C_N
   =
   -\frac{\delta H_C}{\delta N}
   =
   -\frac{\partial H_C}{\partial N}
   +
   D_i\!\left(
   \frac{\partial H_C}{\partial(D_iN)}
   \right);
   }
   \]

4. intégrer par parties lorsque nécessaire afin d'isoler le lapse comme multiplicateur ;

5. vérifier si la densité finale est indépendante de \(N\) et \(D_iN\), modulo contraintes auxiliaires ;

6. seulement alors revenir à
   \[
   \{H[N],H[M]\}.
   \]

Le passage à la dispersion reste interdit.


In [6]:
artifact={
    "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3",
    "final_status":FINAL_STATUS,
    "Ci_Cj_status":"STRUCTURAL_CLOSED_AS_SPATIAL_DIFF_GENERATOR",
    "Cperp_Ci_status":"EXPECTED_BY_SPATIAL_COVARIANCE_BUT_NOT_FULLY_COMPUTED",
    "Cperp_Cperp_status":"BLOCKED_TRUE_LAPSE_SECONDARY_CONSTRAINT_NOT_EXTRACTED",
    "lapse_gradient_obstruction":True,
    "RDD3_status":"OPEN_UNCOMPUTED",
    "gates":GATES,
    "dispersion_ready":False,
    "next":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.1_Lapse_Gradient_Canonicalization_and_True_Secondary_Normal_Constraint_Extraction.ipynb"
}

export_dir=Path("/content/gvh_exports") if Path("/content").exists() else Path.cwd()/"gvh_exports"
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path=export_dir/"gvh_0.3.2.7.3.7.3_hypersurface_algebra_audit.json"
artifact_path.write_text(json.dumps(artifact,indent=2),encoding="utf-8")
print("Artifact:",artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.3_hypersurface_algebra_audit.json


# Conclusion

Le premier audit de l'algèbre produit un résultat important mais différent d'un simple PASS/FAIL.

Le secteur spatial est cohérent :
\[
\boxed{
\{D[\xi],D[\eta]\}=D[[\xi,\eta]]
}
\]
structurellement.

Mais l'analyse directe des objets hérités de 7.7.2.4 montre :
\[
\boxed{
J=J(a_i^{(n)}),\qquad
U=U(a_i^{(n)})
}
\]
avec
\[
a_i^{(n)}=D_i\ln N.
\]

La contrainte secondaire normale doit donc encore être obtenue par une **variation fonctionnelle complète du lapse** avant que le crochet
\[
\{H[N],H[M]\}
\]
puisse être audité honnêtement.

Verdict :
\[
\boxed{\text{PARTIAL PASS}}
\]
avec
\[
\boxed{R_{DD3}\text{ OPEN / UNCOMPUTED}}
\]
et
\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]
